# Gemma 4 E2B. Fine-tuning QLoRA sur Chest X-Ray Pneumonia

**Setup obligatoire :** `Exécution > Modifier le type d'exécution > T4 GPU`

**Pipeline :**
1. Authentification Kaggle (`kaggle.json`)
2. Téléchargement du dataset Kaggle `paultimothymooney/chest-xray-pneumonia`
3. Construction du dataset conversation (image + prompt → réponse label) à partir des dossiers NORMAL / PNEUMONIA, avec une 3e classe `uncertain` pour les images dont le score de confiance qualité est < 0.6
4. Chargement `google/gemma-4-E2B` en 4-bit + adaptateurs QLoRA (PEFT)
5. Fine-tuning avec `trl.SFTTrainer`
6. Évaluation sur le split val
7. Sauvegarde & téléchargement des poids LoRA

> /!\ Prototype pédagogique. Non destiné au diagnostic médical. Toute utilisation clinique requiert validation par un professionnel qualifié.

In [1]:
import os
# Force la gestion des segments expansibles pour éviter la fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
# Vérification rapide de la VRAM disponible
if torch.cuda.is_available():
    print(f"GPU détecté : {torch.cuda.get_device_name(0)}")
    print(f"VRAM totale : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU détecté : NVIDIA GeForce RTX 4070
VRAM totale : 12.88 GB


In [2]:
# ═══════════════════════════════════════════════════════════════════
# STEP 0. Variables d'environnement & utilitaires de reproductibilité
# ═══════════════════════════════════════════════════════════════════
# Doit être exécutée EN PREMIER, avant tout import PyTorch/CUDA.
# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True réduit la fragmentation
# mémoire GPU, utile pour les grands modèles chargés en 4-bit.

import os, random

# Reproductibilité globale
# toutes les cellules suivantes importent la variable SEED plutôt que de coder 42 en dur partout
SEED = 42

def set_all_seeds(seed: int = SEED) -> None:
    # Fixe les seeds Python, NumPy et PyTorch (CPU + GPU)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Déterminisme CUDA, légèrement plus lent mais résultats identiques
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
    except ImportError:
        pass

set_all_seeds(SEED)

# Réduit fragmentation mémoire GPU
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Seed globale fixée à {SEED}')
print('Variables CUDA configurées.')


Seed globale fixée à 42
Variables CUDA configurées.


In [3]:
# ════════════════════════════════════
# STEP 1. Installation des dépendances
# ════════════════════════════════════
# Packages nécessaires :
#   transformers         → support architecture Gemma 4
#   accelerate           → dispatching multi-GPU et mixed-precision
#   bitsandbytes         → quantification 4-bit NF4 (QLoRA)
#   peft                 → LoRA / QLoRA (adaptateurs légers)
#   trl                  → SFTTrainer (fine-tuning supervisé simplifié)
#   pillow               → chargement et conversion des images
#   scikit-learn         → métriques d'évaluation (accuracy, F1, rapport)
#   datasets             → format Dataset HuggingFace pour le trainer
#   kaggle               → téléchargement du dataset chest-xray-pneumonia

!pip install -q -U transformers accelerate "bitsandbytes>=0.46.1" peft trl "pillow<12" scikit-learn datasets kaggle

print('Installation OK')

Installation OK


In [4]:
# ═══════════════════════════════
# STEP 2. Authentification Kaggle
# ═══════════════════════════════
# Uploadez votre fichier kaggle.json (disponible sur https://www.kaggle.com/settings > API Tokens > Create New Token)

import os
from pathlib import Path

# Définition du chemin vers ton fichier actuel
kaggle_file = Path("../kaggle.json")

if kaggle_file.exists():
    # On indique à l'API Kaggle que son dossier de configuration est le dossier parent
    os.environ['KAGGLE_CONFIG_DIR'] = str(kaggle_file.parent.resolve())
    print(f"Authentification Kaggle configurée depuis : {kaggle_file.resolve()}")
else:
    raise FileNotFoundError(
        f"Le fichier {kaggle_file.resolve()} est introuvable. "
        "Vérifiez l'emplacement de votre fichier kaggle.json par rapport au script."
    )

Authentification Kaggle configurée depuis : C:\Users\yanni\Downloads\Kim\kaggle.json


In [5]:
# ══════════════════════════════════════════════════════════════
# STEP 3. Téléchargement du dataset Chest X-Ray Pneumonia (Kaggle)
# ══════════════════════════════════════════════════════════════
# Dataset : paultimothymooney/chest-xray-pneumonia
# Structure attendue après décompression : chest_xray/{train,test,val}/{NORMAL,PNEUMONIA}/*.jpeg
# Contrairement au pipeline RSNA d'origine, ce dataset n'a pas besoin de CSV externe :
# le label (NORMAL/PNEUMONIA) est directement porté par le nom du dossier parent de chaque image.

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/chest_xray --unzip -q

from pathlib import Path
from collections import Counter

root = Path('/content/chest_xray')
IMG_EXTS = {'.jpeg', '.jpg', '.png'}

# Mapping dossier Kaggle → label utilisé dans le pipeline
# 'suspected_opacity' reprend la convention du notebook d'origine (RSNA) pour les cas pathologiques
LABEL_MAP = {
    'NORMAL': 'normal',
    'PNEUMONIA': 'suspected_opacity',
}

all_imgs = [p for p in root.rglob('*') if p.suffix.lower() in IMG_EXTS]
print(f'{len(all_imgs)} images trouvées sur disque')

# Construction de la liste de cas (équivalent du df RSNA, mais sans CSV)
# case_id = nom de fichier sans extension (préfixé par le split Kaggle pour éviter les collisions train/test/val)
cases_raw = []
for p in all_imgs:
    folder = p.parent.name.upper()
    if folder not in LABEL_MAP:
        continue
    kaggle_split = p.parent.parent.name.lower()  # train / test / val
    case_id = f'{kaggle_split}_{p.stem}'
    cases_raw.append({
        'case_id': case_id,
        'label': LABEL_MAP[folder],
        'image_path': str(p),
    })

print(f'{len(cases_raw)} cas valides indexés (variable cases_raw)')
print('Distribution :', Counter(c['label'] for c in cases_raw))


Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
17568 images trouvées sur disque
17568 cas valides indexés (variable cases_raw)
Distribution : Counter({'suspected_opacity': 12819, 'normal': 4749})


In [6]:
# ═════════════════════════════════════════════════════════════════════
# STEP 4. Déduplication + score de confiance qualité + split train/val
# ═════════════════════════════════════════════════════════════════════
#  1. Déduplique sur (case_id, label)
#  2. Calcule un score de confiance heuristique [0,1] par image (proxy qualité :
#     contraste + netteté, normalisé sur l'ensemble du dataset). Le dataset
#     chest-xray-pneumonia n'a pas de label de confiance clinique réel ; ce score
#     sert de proxy pédagogique pour repérer les images les moins exploitables.
#  3. Toute image dont la confiance normalisée est sous le 15e percentile de la
#     distribution est reclassée 'uncertain' (seuil calculé dynamiquement ; la
#     classe d'origine normal/suspected_opacity est conservée dans 'true_label')
#  4. Construit un split train/val STRATIFIÉ et ÉQUILIBRÉ sur 3 classes
#     (normal / suspected_opacity / uncertain), avec random.seed(SEED) pour reproductibilité

import os
import random
import numpy as np
from collections import Counter
from PIL import Image, ImageFilter

# --- SÉCURITÉ VS CODE : Déclaration des variables globales si manquantes ---
if 'SEED' not in locals() and 'SEED' not in globals():
    SEED = 42
    print(f"⚠️ SEED n'était pas défini dans l'environnement. Initialisé à : {SEED}")
else:
    # Récupère la SEED existante globale
    SEED = globals().get('SEED', locals().get('SEED', 42))

# Vérification de la présence de la donnée brute
if 'cases_raw' not in locals() and 'cases_raw' not in globals():
    raise NameError("La variable 'cases_raw' est introuvable. Assurez-vous d'avoir exécuté la cellule de l'Étape 2/3.")

# 1. Déduplication + filtrage des fichiers parasites (ex: __MACOSX, ._fichier)
seen = set()
cases = []
n_skipped_junk = 0
for c in cases_raw:
    path = c['image_path']
    basename = os.path.basename(path)
    if '__MACOSX' in path or basename.startswith('._') or basename.startswith('.'):
        n_skipped_junk += 1
        continue
    key = (c['case_id'], c['label'])
    if key in seen:
        continue
    seen.add(key)
    cases.append(c)

print(f'Après déduplication : {len(cases)} cas (was {len(cases_raw)})')
if n_skipped_junk:
    print(f'   dont {n_skipped_junk} fichiers parasites ignorés (__MACOSX / ._*)')
print('Distribution (label dossier) : International', Counter(c['label'] for c in cases))

# 2. Score de confiance qualité (contraste + netteté)
QUALITY_SIZE = 64  # downscale pour accélérer le calcul

def quality_metrics(image_path: str):
    """Retourne (contraste, netteté) bruts pour une image, sur une version réduite.
    Retourne None si le fichier n'est pas une image lisible."""
    try:
        # Utilisation d'un gestionnaire de contexte 'with' pour libérer le descripteur de fichier immédiatement
        with Image.open(image_path) as img:
            img_gray = img.convert('L').resize((QUALITY_SIZE, QUALITY_SIZE))
            arr = np.asarray(img_gray, dtype=np.float32)
            contrast = float(arr.std())
            
            edges = np.asarray(img_gray.filter(ImageFilter.FIND_EDGES), dtype=np.float32)
            sharpness = float(edges.std())
            return contrast, sharpness
    except Exception as e:
        print(f'    Image illisible ignorée : {image_path} ({e})')
        return None

print(f'Calcul du score de confiance qualité sur {len(cases)} images (Optimisé avec fermeture de flux)...')
raw_metrics_all = [quality_metrics(c['image_path']) for c in cases]

# Ne garder que les cas dont l'image a pu être lue, pour rester synchronisé
paired = [(c, m) for c, m in zip(cases, raw_metrics_all) if m is not None]
n_unreadable = len(cases) - len(paired)
if n_unreadable:
    print(f'{n_unreadable} image(s) illisible(s) exclue(s) du dataset.')
cases = [c for c, m in paired]
raw_metrics = [m for c, m in paired]

contrasts   = np.array([m[0] for m in raw_metrics])
sharpnesses = np.array([m[1] for m in raw_metrics])

# Normalisation min-max indépendante de chaque métrique, puis moyenne
def min_max_norm(x: np.ndarray) -> np.ndarray:
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-8:
        return np.full_like(x, 0.5)
    return (x - lo) / (hi - lo)

confidence_scores = (min_max_norm(contrasts) + min_max_norm(sharpnesses)) / 2.0

# 3. Reclassification 'uncertain' pour les confidences faibles
CONFIDENCE_PERCENTILE = 15
CONFIDENCE_THRESHOLD = float(np.percentile(confidence_scores, CONFIDENCE_PERCENTILE))
print(f'Seuil de confiance calculé : {CONFIDENCE_THRESHOLD:.4f} (percentile {CONFIDENCE_PERCENTILE})')

n_reclassified = 0
for c, score in zip(cases, confidence_scores):
    c['confidence']  = round(float(score), 4)
    c['true_label']  = c['label']  # conserve le label d'origine
    if c['confidence'] < CONFIDENCE_THRESHOLD:
        c['label'] = 'uncertain'
        n_reclassified += 1

print(f'{n_reclassified} cas reclassés "uncertain"')
print('Distribution (3 classes) :', Counter(c['label'] for c in cases))

# 4. Paramètres du split stratifié
N_PER_CLASS_TRAIN = 150
N_PER_CLASS_VAL   = 50
ALL_LABELS = ['normal', 'suspected_opacity', 'uncertain']

# Garantit un split identique à chaque exécution du notebook
random.seed(SEED)
train_cases, val_cases = [], []
for label in ALL_LABELS:
    subset = [c for c in cases if c['label'] == label]
    if len(subset) == 0:
        print(f'   Classe "{label}" absente du dataset.')
        continue
    random.shuffle(subset)
    
    # Sécurité mathématique pour éviter d'obtenir un n_train négatif si le dataset est trop petit
    n_train = min(N_PER_CLASS_TRAIN, max(0, len(subset) - N_PER_CLASS_VAL))
    n_val   = min(N_PER_CLASS_VAL,   len(subset) - n_train)
    
    train_cases.extend(subset[:n_train])
    val_cases.extend(subset[n_train:n_train + n_val])
    print(f'   {label:20s} → train={n_train}, val={n_val}  (dispo={len(subset)})')

# Remélange listes avec même seed pour éviter tout ordre résiduel
random.seed(SEED)
random.shuffle(train_cases)
random.shuffle(val_cases)

print(f'\nTrain total : {len(train_cases)} | Val total : {len(val_cases)}')
print('Train dist :', Counter(c['label'] for c in train_cases))
print('Val dist   :', Counter(c['label'] for c in val_cases))

Après déduplication : 5856 cas (was 17568)
   dont 5856 fichiers parasites ignorés (__MACOSX / ._*)
Distribution (label dossier) : International Counter({'suspected_opacity': 4273, 'normal': 1583})
Calcul du score de confiance qualité sur 5856 images (Optimisé avec fermeture de flux)...
Seuil de confiance calculé : 0.3613 (percentile 15)
879 cas reclassés "uncertain"
Distribution (3 classes) : Counter({'suspected_opacity': 3396, 'normal': 1581, 'uncertain': 879})
   normal               → train=150, val=50  (dispo=1581)
   suspected_opacity    → train=150, val=50  (dispo=3396)
   uncertain            → train=150, val=50  (dispo=879)

Train total : 450 | Val total : 150
Train dist : Counter({'normal': 150, 'suspected_opacity': 150, 'uncertain': 150})
Val dist   : Counter({'suspected_opacity': 50, 'uncertain': 50, 'normal': 50})


In [7]:
# ════════════════════════════════════════
# STEP 5. Authentification HuggingFace Hub
# ════════════════════════════════════════
# Nécessaire pour télécharger google/gemma-4-E2B (modèle à accès restreint)
# Obtenez un token sur : https://huggingface.co/settings/tokens
# Vérifiez également que vous avez accepté les conditions d'utilisation sur : https://huggingface.co/google/gemma-4-E2B

from huggingface_hub import login
login() 


c:\Users\yanni\Downloads\Kim\Solution-Delivery_DS-7B_ARVI-RX-Medgemma-Test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# ══════════════════════════════════════════════════
# STEP 6. Chargement Gemma 4 E2B (FIX : Désencapsulation)
# ══════════════════════════════════════════════════

import torch
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig, AutoProcessor
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. INITIALISER LE PROCESSOR (Manquant dans ton bloc)
print("Chargement du processor...")
processor = AutoProcessor.from_pretrained('google/gemma-4-E2B')

# 1. Charger le modèle normalement
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForImageTextToText.from_pretrained(
    'google/gemma-4-E2B',
    quantization_config=bnb_config,
    device_map={"": 0},
    attn_implementation="eager",
)

# 2. FONCTION DE DÉSENCAPSULATION (Le Fix)
def unwrap_gemma4_layers(module):
    """
    Parcourt le modèle et remplace 'Gemma4ClippableLinear' 
    par la couche 'linear' interne (qui est un Linear4bit supporté).
    """
    for name, child in module.named_children():
        if child.__class__.__name__ == 'Gemma4ClippableLinear' and hasattr(child, 'linear'):
            # On remplace l'enveloppe non supportée par la couche linéaire interne
            setattr(module, name, child.linear)
        else:
            unwrap_gemma4_layers(child)

print("Application du patch de compatibilité PEFT...")
unwrap_gemma4_layers(model)

# 3. Préparer pour l'entraînement
# model = prepare_model_for_kbit_training(model)

model.gradient_checkpointing_enable() # Active le checkpointing pour économiser la VRAM
model.enable_input_require_grads()    # Obligatoire pour LoRA en 4-bit

# 4. Configurer LoRA (maintenant PEFT verra des Linear4bit standards)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
print("Modèle prêt pour le fine-tuning !")
model.print_trainable_parameters()

# model = get_peft_model(model, lora_config)
# print("Modèle chargé avec succès.")
# print('Device de calcul ancré :', next(model.parameters()).device)

Chargement du processor...


c:\Users\yanni\Downloads\Kim\Solution-Delivery_DS-7B_ARVI-RX-Medgemma-Test\.venv\Lib\site-packages\transformers\modeling_utils.py:5095: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.empty(int(byte_count // 2), dtype=torch.float16, device=device, requires_grad=False)
Loading weights: 100%|██████████| 1951/1951 [00:08<00:00, 242.21it/s] 


Application du patch de compatibilité PEFT...
Modèle prêt pour le fine-tuning !
trainable params: 14,929,920 || all params: 5,119,227,424 || trainable%: 0.2916


In [9]:
# ═══════════════════════════════
# STEP 7. Construction du dataset
# ═══════════════════════════════
# Chaque exemple est formaté en conversation Gemma (ChatML-like):
#   <start_of_turn>system → instructions du système
#   <start_of_turn>user → token image + prompt de classification
#   <start_of_turn>model → réponse JSON cible (supervisée)

from datasets import Dataset
from PIL import Image
import json

# Token image attendu par le processor (corrigé : image_token, pas boi_token)
BOI = processor.image_token
print(f'Token image utilisé : {repr(BOI)}')

# Prompts système et utilisateur
SYSTEM_PROMPT = (
    'You are an educational radiology assistant for engineering students. '
    'You are not a clinician. Analyze the frontal chest X-ray and return '
    'only the classification label as valid JSON.'
)

USER_PROMPT = (
    'Classify this frontal chest X-ray. '
    'Return ONLY valid JSON with exactly these fields:\n'
    "{\n"
    '  "image_quality": "good | limited | poor",\n'
    '  "predicted_class": "normal | suspected_opacity | uncertain",\n'
    '  "confidence": 0.0,\n'
    '  "visual_evidence": ["short factual observation"],\n'
    '  "justification": "2 to 4 cautious sentences strictly based on visible evidence",\n'
    '  "limitations": ["image quality", "projection", "lack of clinical context"],\n'
    '  "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image."\n'
    "}\n"
    'Rules:\n'
    '- Use "suspected_opacity" when there is a visible pulmonary opacity or consolidation.\n'
    '- Use "normal" when no significant abnormality is visible.\n'
    '- Use "uncertain" when image quality is poor, the view or findings are ambiguous.\n'
    '- If JSON cannot be guaranteed, return "uncertain"'
)

# Confidence cible par classe (fallback)
LABEL_CONFIDENCE = {
    'normal':            0.95,
    'suspected_opacity': 0.90,
    'uncertain':         0.60,
}

def build_conversation(case: dict) -> str:
    """Formate un cas en texte de conversation Gemma multi-tour.

    Format :
        <start_of_turn>system\n...\n<end_of_turn>\n
        <start_of_turn>user\n<|image|>\n...\n<end_of_turn>\n
        <start_of_turn>model\n{JSON}\n<end_of_turn>
    """
    answer = json.dumps({
        'predicted_class': case['label'],
        'confidence': case.get('confidence', LABEL_CONFIDENCE.get(case['label'], 0.80)),
    })
    text = (
        f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
        f'<start_of_turn>user\n{BOI}\n{USER_PROMPT}<end_of_turn>\n'
        f'<start_of_turn>model\n{answer}<end_of_turn>'
    )
    return text

def preprocess_lazy(cases_list: list) -> dict:
    """Construit les listes parallèles sans charger les images en RAM."""
    texts, image_paths = [], []
    for case in cases_list:
        texts.append(build_conversation(case))
        image_paths.append(case['image_path'])
    return {'text': texts, 'image_path': image_paths}

print('Construction du dataset train (Lazy)...')
train_data = preprocess_lazy(train_cases)
print('Construction du dataset val (Lazy)...')
val_data = preprocess_lazy(val_cases)

# On passe 'image_path' au lieu de l'objet Image PIL
train_dataset = Dataset.from_dict({'text': train_data['text'], 'image_path': train_data['image_path']})
val_dataset = Dataset.from_dict({'text': val_data['text'],   'image_path': val_data['image_path']})

print(f'Train dataset : {len(train_dataset)} exemples indexés (0 Mo de RAM ajoutés)')
print(f'Val dataset   : {len(val_dataset)} exemples indexés')
print(f'Train dataset : {len(train_dataset)} exemples indexés (0 Mo de RAM utilisés)')
print(f'Val dataset   : {len(val_dataset)} exemples indexés')
# Affichage d'un exemple formaté
print('\nExemple texte (1er cas train) :')
print(train_dataset[0]['text'])

print('boi_token   :', BOI)
print('image_token :', processor.image_token)


Token image utilisé : '<|image|>'
Construction du dataset train (Lazy)...
Construction du dataset val (Lazy)...
Train dataset : 450 exemples indexés (0 Mo de RAM ajoutés)
Val dataset   : 150 exemples indexés
Train dataset : 450 exemples indexés (0 Mo de RAM utilisés)
Val dataset   : 150 exemples indexés

Exemple texte (1er cas train) :
<start_of_turn>system
You are an educational radiology assistant for engineering students. You are not a clinician. Analyze the frontal chest X-ray and return only the classification label as valid JSON.<end_of_turn>
<start_of_turn>user
<|image|>
Classify this frontal chest X-ray. Return ONLY valid JSON with exactly these fields:
{
  "image_quality": "good | limited | poor",
  "predicted_class": "normal | suspected_opacity | uncertain",
  "confidence": 0.0,
  "visual_evidence": ["short factual observation"],
  "justification": "2 to 4 cautious sentences strictly based on visible evidence",
  "limitations": ["image quality", "projection", "lack of clinica

In [10]:
# ═════════════════════════════════════════
# STEP 8. Collator multimodal personnalisé 
# ═════════════════════════════════════════

import torch
from dataclasses import dataclass
from typing import Any, Dict, List
from PIL import Image

@dataclass
class MedGemmaCollator:
    """Collator multimodal pour MedGemma/Gemma3 avec gestion explicite du jeton <image>."""
    processor: Any
    max_length: int = 512

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        texts = [f['text'] for f in features]
        images = [Image.open(f['image_path']).convert('RGB') for f in features]

        # Le processor doit gérer la tokenisation du texte ET l'encodage des images
        # Si <|image|> est dans le texte, il doit être reconnu par le tokenizer
        batch_enc = self.processor(
            text=texts,
            images=images,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=self.max_length,
        )

        # 3. Préparation des labels pour l'entraînement
        # Le but est de masquer (mettre à -100) tout ce qui n'est pas la réponse du modèle
        labels = batch_enc['input_ids'].clone()
        
        # On masque le padding (en supposant que pad_id est -1 ou 0 selon tokenizer)
        pad_id = self.processor.tokenizer.pad_token_id
        labels[labels == pad_id] = -100

        # 4. Masquage dynamique du prompt (System + User)
        # On ne veut pas que le modèle apprenne à prédire les instructions, 
        # seulement la réponse JSON.
        model_turn_token = self.processor.tokenizer.encode(
            '<start_of_turn>model\n', add_special_tokens=False
        )
        
        # Parcourir chaque exemple du batch pour masquer le prompt
        for i in range(len(labels)):
            # Trouver la position du marqueur '<start_of_turn>model\n'
            # (Astuce: on cherche la séquence de tokens dans les input_ids)
            ids = batch_enc['input_ids'][i]
            
            # Recherche naïve de la séquence
            for j in range(len(ids) - len(model_turn_token)):
                if torch.equal(ids[j : j + len(model_turn_token)], torch.tensor(model_turn_token, device=ids.device)):
                    # Masquer tout ce qui est avant la fin du marqueur "model"
                    labels[i, :j + len(model_turn_token)] = -100
                    break
        
        batch_enc['labels'] = labels
        return batch_enc

# Initialisation du collator avec le processeur chargé
collator = MedGemmaCollator(processor=processor)
print('Collator multimodal prêt.')

Collator multimodal prêt.


In [ ]:
# ═══════════════════════════════════════════════════
# STEP 9. Configuration des arguments d'entraînement
# ═══════════════════════════════════════════════════
from trl import SFTConfig, SFTTrainer  # SFTConfig au lieu de TrainingArguments

OUTPUT_DIR = '/content/gemma4_chestxray_qlora'

use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
print(f'Mode précision : {"bf16" if use_bf16 else "fp16"}')

steps_per_epoch = len(train_dataset) // (1 * 4) # batch_size=1, gradient_accumulation_steps=4
total_steps = steps_per_epoch * 2
warmup_steps = max(1, int(total_steps * 0.05))
print(f'Steps/epoch={steps_per_epoch}, total={total_steps}, warmup={warmup_steps}')

checkpoint_every = max(50, steps_per_epoch // 2)
print(f'Checkpoint tous les {checkpoint_every} steps')

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=warmup_steps,
    optim='adamw_8bit',
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=checkpoint_every,
    save_strategy='steps',
    save_steps=checkpoint_every,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    dataloader_num_workers=0,
    remove_unused_columns=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    max_grad_norm=0.3,
    seed=SEED,
    data_seed=SEED,
    weight_decay=0.01,
    dataset_kwargs={'skip_prepare_dataset': True},  # ← clé : désactive le pré-traitement auto de SFTTrainer
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator
)
print('Trainer configuré')
print(f'Steps effectifs/epoch : {steps_per_epoch}')

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Mode précision : bf16
Steps/epoch=112, total=224, warmup=11
Checkpoint tous les 56 steps
Trainer configuré
Steps effectifs/epoch : 112


In [ ]:
# ══════════════════════════
# STEP 10. Fine-tuning QLoRA
# ══════════════════════════
# Cette cellule lance l'entraînement. Avant de démarrer :
#   1. Vide le cache GPU pour libérer la VRAM fragmentée
#   2. Affiche la VRAM libre pour vérifier qu'on a assez de marge
#   3. Supprime les warnings de dépréciation du tokenizer (tokens PAD/BOS/EOS) qui sont inoffensifs mais polluent les logs d'entraînement

import gc
import warnings

# Alignement config modèle ↔ tokenizer
# Le tokenizer MedGemma peut avoir des tokens spéciaux différents de ceux enregistrés dans model.config / model.generation_config
# Transformers détecte cet écart et le corrige tout seul (avec un warning). On le fait explicitement ici pour éviter le message
tok = processor.tokenizer
for config_obj in [model.config, getattr(model, 'generation_config', None)]:
    if config_obj is None:
        continue
    if hasattr(config_obj, 'pad_token_id') and tok.pad_token_id is not None:
        config_obj.pad_token_id = tok.pad_token_id
    if hasattr(config_obj, 'bos_token_id') and tok.bos_token_id is not None:
        config_obj.bos_token_id = tok.bos_token_id
    if hasattr(config_obj, 'eos_token_id') and tok.eos_token_id is not None:
        config_obj.eos_token_id = tok.eos_token_id

# Supprime  warnings résiduels de tokens spéciaux pendant entraînement
warnings.filterwarnings('ignore', message='.*PAD/BOS/EOS tokens.*')
warnings.filterwarnings('ignore', message='.*image_token.*boi_token.*')

# Nettoyage mémoire GPU
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM libre avant training : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# Lancement de l'entraînement
print('Démarrage du fine-tuning QLoRA...')
trainer.train()
print('Fine-tuning terminé !')


In [ ]:
# ══════════════════════════════════════════════════════════
# STEP 11. Sauvegarde et téléchargement des adaptateurs LoRA
# ══════════════════════════════════════════════════════════

import shutil
from pathlib import Path

LORA_SAVE_PATH = Path('./gemma4_chestxray_lora_adapters').resolve()

def save_lora():
    LORA_SAVE_PATH.mkdir(parents=True, exist_ok=True)

    # sauvegarde
    model.save_pretrained(LORA_SAVE_PATH)
    processor.save_pretrained(LORA_SAVE_PATH)
    print(f'Adaptateurs LoRA sauvegardés dans : {LORA_SAVE_PATH}')    

    # Vérification contenu
    if not LORA_SAVE_PATH.exists() or len(list(LORA_SAVE_PATH.iterdir())) == 0:
        raise RuntimeError("Échec: le dossier LoRA est vide.")

save_lora()

# Liste et taille des fichiers sauvegardés
print('\nFichiers sauvegardés :')
for p in sorted(LORA_SAVE_PATH.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:<40s} {size_mb:.2f} MB')

# Compression locale en ZIP
archive_path = Path('./lora_adapters_backup')
shutil.make_archive(str(archive_path), 'zip', LORA_SAVE_PATH)

print(f'\nCompression terminée : {archive_path}.zip')
print('Fichiers prêts à être copiés ou déplacés manuellement.')

In [ ]:
# ═════════════════════════════════════════════════════════════════
# STEP 12. Merge des adaptateurs LoRA + évaluation sur le split val
# ═════════════════════════════════════════════════════════════════
#   1. Libère le modèle QLoRA de la VRAM
#   2. Recharge le modèle de base en bf16/fp16 et fusionne les poids LoRA
#   3. Évalue sur val_cases avec décodage JSON et fallback textuel
#   4. Affiche accuracy, macro F1 et rapport de classification

import re, json, gc
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report
from peft import PeftModel
from transformers import AutoModelForImageTextToText

LORA_SAVE_PATH = '/content/gemma4_chestxray_lora_adapters'

# 1. Libération VRAM
# pour éviter Out Of Memory GPU
del model
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM libre après del : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

# 2. Rechargement base + merge LoRA
# charge en bf16 (ou fp16 sur T4) pour maximiser précision en inférence
# merge_and_unload() fusionne poids LoRA dans modèle de base et supprime couches PEFT
use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f'Dtype inférence : {dtype}')

base = AutoModelForImageTextToText.from_pretrained(
    'google/gemma-4-E2B',
    torch_dtype=dtype,
    device_map='auto',
)
merged_model = PeftModel.from_pretrained(base, LORA_SAVE_PATH)
merged_model = merged_model.merge_and_unload()
merged_model.eval()
merged_model.config.use_cache = True  # Active le KV-cache pour l'inférence
print('Modèle mergé prêt')

# 3. Prompt d'évaluation
# EVAL_PROMPT: texte passé au processor en inférence (pas de réponse)
# Il doit correspondre exactement au format vu pendant le training
EVAL_PROMPT = (
    f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
    f'<start_of_turn>user\n{BOI}\n{USER_PROMPT}<end_of_turn>\n'
    f'<start_of_turn>model\n'
)

# Clés à passer à generate() (on exclut les champs non-reconnus)
GENERATE_KEYS = {'input_ids', 'attention_mask', 'pixel_values'}
VALID_LABELS  = {'normal', 'suspected_opacity', 'uncertain'}  # 'uncertain' reste une sortie possible du modèle

def evaluate_case(image_path: str) -> tuple:
    """Prédit la classe d'un cas radiologique.

    Retourne (predicted_class, raw_text) avec un fallback textuel si le
    JSON est malformé ou si predicted_class n'est pas dans VALID_LABELS.
    """
    image = Image.open(image_path).convert('RGB')
    inputs = processor(text=EVAL_PROMPT, images=[image], return_tensors='pt')
    inputs = {k: v.to(merged_model.device) for k, v in inputs.items() if k in GENERATE_KEYS}

    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=64, # Suffisant pour {"predicted_class":...,"confidence":...}
            do_sample=False, # Greedy decoding → déterministe
            use_cache=True,
        )

    # Décode uniquement les tokens générés
    input_len = inputs['input_ids'].shape[-1]
    text = processor.decode(outputs[0][input_len:], skip_special_tokens=True)

    try:
        # Extraction du premier objet JSON dans la réponse
        match = re.search(r'\{.*?\}', text, re.DOTALL)
        result = json.loads(match.group()) if match else {}
        pred = result.get('predicted_class', 'uncertain')
        if pred not in VALID_LABELS:
            pred = 'uncertain'
    except Exception:
        # Fallback: recherche textuelle des labels dans la réponse brute
        text_l = text.lower()
        if 'suspected_opacity' in text_l or 'opacity' in text_l:
            pred = 'suspected_opacity'
        elif 'normal' in text_l:
            pred = 'normal'
        else:
            pred = 'uncertain'

    return pred, text

# 4. Boucle d'évaluation
y_true, y_pred = [], []
for i, case in enumerate(val_cases):
    pred, raw = evaluate_case(case['image_path'])
    y_true.append(case['label'])
    y_pred.append(pred)
    ok = '✓' if pred == case['label'] else '✗'
    print(f'[{i+1:03d}/{len(val_cases)}] {ok} label={case["label"]:20s} pred={pred}')

print('\n=== Résultats validation ===')
print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
print(f'Macro F1 : {f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')
print('\nRapport détaillé :')
print(classification_report(
    y_true, y_pred,
    labels=['normal', 'suspected_opacity', 'uncertain'],
    zero_division=0
))  # 'uncertain' peut apparaître côté prédictions même hors labels d'entraînement


## Notes techniques

**Dataset :** [chest-xray-pneumonia](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) (Kaggle) — images classées par dossier `NORMAL` / `PNEUMONIA`, mappées respectivement vers `normal` / `suspected_opacity`. Une 3e classe `uncertain` est ajoutée automatiquement : tout cas dont le score de confiance qualité (contraste + netteté, normalisé sur le dataset) tombe sous 0.6 est reclassé `uncertain`, quel que soit son label de dossier d'origine.

**Modèle :** `google/gemma-4-E2B` (base pré-entraîné, multimodal texte+image)

**QLoRA :** quantification 4-bit NF4 + adaptateurs LoRA `r=16, alpha=32` sur les couches d'attention et FFN

**Loss :** calculée uniquement sur les tokens de réponse (masquage du prompt utilisateur)

**Format de sortie entraîné :**
```json
{
  "image_quality": "good | limited | poor",
  "predicted_class": "normal | suspected_opacity | uncertain",
  "confidence": 0.0,
  "visual_evidence": ["short factual observation"],
  "justification": "2 to 4 cautious sentences based only on visible evidence",
  "limitations": ["image quality", "projection", "lack of clinical context"],
  "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image."
}
```

**Réutiliser les adaptateurs LoRA :**
```python
from peft import PeftModel
model = AutoModelForImageTextToText.from_pretrained('google/gemma-4-E2B', ...)
model = PeftModel.from_pretrained(model, 'chemin/vers/lora_adapters')
```
